In [59]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

# Build the path to the SQLite database.
database_path = (
    Path.cwd().parent
    / "data"
    / "raw"
    / "MIG_Cement_Records.db"
).resolve()

# Confirm that the database file exists.
print("Database path:", database_path)
print("Database exists:", database_path.exists())

# Stop with a clear error if the file cannot be found.
if not database_path.exists():
    raise FileNotFoundError(
        f"Database not found: {database_path}"
    )

# Open the SQLite database in read-only mode.
connection = sqlite3.connect(
    f"file:{database_path.as_posix()}?mode=ro",
    uri=True
)

# Load the Operations table into a pandas DataFrame.
operations = pd.read_sql_query(
    "SELECT * FROM Operations;",
    connection
)

# Load the Sites reference table.
sites = pd.read_sql_query(
    "SELECT * FROM Sites;",
    connection
)

# Load the CementTypes reference table.
cement_types = pd.read_sql_query(
    "SELECT * FROM CementTypes;",
    connection
)

# Close the database connection.
connection.close()

# Confirm that all three tables loaded successfully.
print("Operations:", operations.shape)
print("Sites:", sites.shape)
print("Cement types:", cement_types.shape)


Database path: C:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\raw\MIG_Cement_Records.db
Database exists: True
Operations: (32880, 11)
Sites: (30, 4)
Cement types: (3, 1)


In [60]:
#Standardise column names


In [61]:
# Create a copy of the Operations DataFrame.
# This prevents accidental modification of the original raw dataset.
operations_clean = operations.copy()

# Create a copy of the Sites reference table.
sites_clean = sites.copy()

# Create a copy of the CementTypes reference table.
cement_types_clean = cement_types.copy()


In [62]:
# 2. Standardise column names

def standardise_column_names(dataframe):
    """
    Standardise DataFrame column names.

    The function:
    - removes leading and trailing spaces;
    - converts names to lowercase;
    - replaces spaces with underscores;
    - replaces hyphens with underscores.
    """

    dataframe.columns = (
        dataframe.columns
        .str.strip()                # Remove leading/trailing spaces
        .str.lower()                # Convert to lowercase
        .str.replace(" ", "_", regex=False)  # Replace spaces with underscores
        .str.replace("-", "_", regex=False)  # Replace hyphens with underscores
    )

    return dataframe


In [63]:
operations_clean = standardise_column_names(operations_clean)
sites_clean = standardise_column_names(sites_clean)
cement_types_clean = standardise_column_names(cement_types_clean)


In [64]:
print("Operations columns:")
print(operations_clean.columns.tolist())

print("\nSites columns:")
print(sites_clean.columns.tolist())

print("\nCement Types columns:")
print(cement_types_clean.columns.tolist())


Operations columns:
['date', 'site_id', 'cement_type', 'planned_pour_tonnes', 'consumed_tonnes', 'opening_inventory_tonnes', 'deliveries_tonnes', 'closing_inventory_tonnes', 'rain_mm', 'avg_temp_c', 'silo_capacity']

Sites columns:
['site_id', 'region', 'silo_capacity', 'behavior']

Cement Types columns:
['cement_type']


In [65]:
#verifying the date column in the Operations table

print("Start date:", operations["date"].min())
print("End date:", operations["date"].max())
print("Invalid dates:", operations["date"].isna().sum())


Start date: 2022-01-01
End date: 2024-12-31
Invalid dates: 0


In [66]:
#checking for missing values in the Operations table
print("Missing values in each column:")
print(operations.isnull().sum())


Missing values in each column:
date                        0
site_id                     0
cement_type                 0
planned_pour_tonnes         0
consumed_tonnes             0
opening_inventory_tonnes    0
deliveries_tonnes           0
closing_inventory_tonnes    0
rain_mm                     0
avg_temp_c                  0
silo_capacity               0
dtype: int64


In [67]:
#checking for duplicate rows in the Operations table

print("Duplicate Rows:", operations.duplicated().sum())


Duplicate Rows: 0


In [68]:
# Every combination of date, site and cement type
# should appear only once.

operations = operations.drop_duplicates(
    subset=[
        "date",
        "site_id",
        "cement_type"
    ]
)


#Remove invalid site 

In [69]:
# Keep only records whose site_id exists in the Sites table.

operations = operations[
    operations["site_id"].isin(
        sites["site_id"]
    )
]


#Remove Invalid Cement Types

In [70]:
# Keep only valid cement types.

operations = operations[
    operations["cement_type"].isin(
        cement_types["cement_type"]
    )
]


#Remove Negative Values

In [71]:
# These columns should never contain negative values.

non_negative_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "silo_capacity"
]

# Remove rows containing negative values.

for column in non_negative_columns:
    operations = operations[
        operations[column] >= 0
    ]


#Validate Inventory Balance

In [72]:
# Calculate the expected closing inventory using
# Opening Inventory + Deliveries - Consumption.

operations["calculated_closing_inventory"] = (
    operations["opening_inventory_tonnes"]
    + operations["deliveries_tonnes"]
    - operations["consumed_tonnes"]
)

# Calculate the difference between the reported
# and calculated closing inventory.

operations["inventory_difference"] = (
    operations["closing_inventory_tonnes"]
    - operations["calculated_closing_inventory"]
)

# Round the inventory values to remove tiny
# floating-point precision errors.

operations["closing_inventory_tonnes"] = (
    operations["calculated_closing_inventory"]
    .round(2)
)


#Round Numerical Values

In [73]:
# Round all numerical columns to two decimal places.

float_columns = operations.select_dtypes(
    include="float64"
).columns

operations[float_columns] = (
    operations[float_columns]
    .round(2)
)


#Merge the Site Information

In [74]:
# Merge operational records with the Sites table
# to include region and site behaviour.

clean_data = operations.merge(
    sites[
        [
            "site_id",
            "region",
            "behavior"
        ]
    ],
    on="site_id",
    how="left"
)


#Create New Variables

In [75]:
# Calculate the total cement available each day.

clean_data["available_inventory_tonnes"] = (
    clean_data["opening_inventory_tonnes"]
    + clean_data["deliveries_tonnes"]
)

# Calculate the amount of planned demand
# that was not fulfilled.

clean_data["unmet_demand_tonnes"] = (
    clean_data["planned_pour_tonnes"]
    - clean_data["consumed_tonnes"]
).clip(lower=0)

# Identify stockout events.

clean_data["stockout_flag"] = (
    clean_data["closing_inventory_tonnes"] == 0
).astype(int)

# Identify whether the planned pour
# was successfully completed.

clean_data["pour_ready_flag"] = (
    clean_data["consumed_tonnes"]
    >= clean_data["planned_pour_tonnes"]
).astype(int)


#Fill missing operational value

In [76]:
# Define the main operational measurement columns.
operational_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes"
]


# Process every operational column separately.
for column in operational_columns:

    # Calculate the median for each site and cement-type combination.
    group_median = (
        operations_clean
        .groupby(
            [
                "site_id",
                "cement_type"
            ]
        )[column]
        .transform("median")
    )

    # Replace missing values with the appropriate group median.
    operations_clean[column] = (
        operations_clean[column]
        .fillna(group_median)
    )


#Use the Sites table as the source of truth for silo capacity

In [77]:
# Select only the site ID and silo capacity from the Sites table.
site_capacity_reference = (
    sites_clean[
        [
            "site_id",
            "silo_capacity"
        ]
    ]

    # Make sure each site appears only once.
    .drop_duplicates(
        subset=["site_id"]
    )

    # Rename the capacity field to avoid confusion during the merge.
    .rename(
        columns={
            "silo_capacity":
            "reference_silo_capacity"
        }
    )
)


# Join the trusted site capacity to every operational record.
operations_clean = operations_clean.merge(
    site_capacity_reference,
    on="site_id",
    how="left",

    # Confirm that many Operations rows map to one Sites row.
    validate="many_to_one"
)
#Check whether the two capacity values disagree:

# Compare the Operations capacity with the Sites reference capacity.
capacity_mismatch = (
    operations_clean["silo_capacity"]
    != operations_clean[
        "reference_silo_capacity"
    ]
)


# Display the number of differences.
print(
    "Silo-capacity mismatches:",
    capacity_mismatch.sum()
)
#Replace the Operations value:

# Use the Sites-table capacity when it is available.
# Otherwise, retain the original Operations value.
operations_clean["silo_capacity"] = (
    operations_clean[
        "reference_silo_capacity"
    ]
    .fillna(
        operations_clean["silo_capacity"]
    )
)


# Remove the temporary reference column.
operations_clean = operations_clean.drop(
    columns=[
        "reference_silo_capacity"
    ]
)


Silo-capacity mismatches: 0


#Validate the inventory balance

In [78]:
# Calculate what closing inventory should be according to the business rule:
# opening inventory + deliveries - consumed amount.
operations_clean[
    "calculated_closing_inventory"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    + operations_clean[
        "deliveries_tonnes"
    ]
    - operations_clean[
        "consumed_tonnes"
    ]
)


# Calculate the difference between reported and expected closing inventory.
operations_clean[
    "inventory_balance_difference"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ]
    - operations_clean[
        "calculated_closing_inventory"
    ]
).round(4)
#Define a rounding tolerance:

# Allow very small differences caused by decimal rounding.
inventory_tolerance = 0.011


# Identify records where the difference is larger than the tolerance.
inventory_mismatch_mask = (
    operations_clean[
        "inventory_balance_difference"
    ].abs()
    > inventory_tolerance
)


# Display the number of material mismatches.
print(
    "Material inventory mismatches:",
    inventory_mismatch_mask.sum()
)
#Correct only material errors:

# Replace the reported closing inventory only where
# the difference exceeds the accepted tolerance.
operations_clean.loc[
    inventory_mismatch_mask,
    "closing_inventory_tonnes"
] = operations_clean.loc[
    inventory_mismatch_mask,
    "calculated_closing_inventory"
]


Material inventory mismatches: 0


In [79]:
# Define columns that should use two decimal places.
tonnage_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes"
]


# Round all tonnage measurements to two decimal places.
operations_clean[tonnage_columns] = (
    operations_clean[
        tonnage_columns
    ].round(2)
)


#Standardise text values

In [80]:
# Convert site IDs to pandas string type.
operations_clean["site_id"] = (
    operations_clean["site_id"]
    .astype("string")
    .str.strip()   # Remove spaces before or after each value.
    .str.upper()   # Ensure values use uppercase, such as SITE_001.
)


# Convert cement types to consistent uppercase text.
operations_clean["cement_type"] = (
    operations_clean["cement_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Clean site IDs in the Sites reference table.
sites_clean["site_id"] = (
    sites_clean["site_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# Clean region names.
sites_clean["region"] = (
    sites_clean["region"]
    .astype("string")
    .str.strip()
    .str.title()   # Convert values to title case, such as North or South.
)


# Clean operating-behaviour values.
sites_clean["behavior"] = (
    sites_clean["behavior"]
    .astype("string")
    .str.strip()
    .str.lower()   # Use lowercase values such as aggressive.
)


# Clean cement types in the CementTypes reference table.
cement_types_clean["cement_type"] = (
    cement_types_clean["cement_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)


#Convert columns to the correct data types

In [81]:
# Convert the date column from text to pandas datetime.
# Invalid dates will become NaT, which means "Not a Time".
operations_clean["date"] = pd.to_datetime(
    operations_clean["date"],
    errors="coerce"
)


# Define the Operations columns that should contain numeric values.
numeric_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity"
]


# Convert every listed column to a numeric data type.
for column in numeric_columns:

    operations_clean[column] = pd.to_numeric(
        operations_clean[column],

        # Invalid values will become NaN instead of causing an error.
        errors="coerce"
    )


# Convert site-level silo capacity to numeric.
sites_clean["silo_capacity"] = pd.to_numeric(
    sites_clean["silo_capacity"],
    errors="coerce"
)


#Remove records missing critical identifiers

In [82]:
# Define columns that every operational record must contain.
critical_columns = [
    "date",
    "site_id",
    "cement_type"
]


# Store the row count before removing incomplete records.
before_rows = len(operations_clean)


# Remove rows where any critical identifier is missing.
operations_clean = operations_clean.dropna(
    subset=critical_columns
)


# Store the row count after cleansing.
after_rows = len(operations_clean)


# Report how many rows were removed.
print(
    "Rows removed because of missing critical fields:",
    before_rows - after_rows
)


Rows removed because of missing critical fields: 0


#Validate site IDs and cement types

In [83]:
# Create a set of valid site IDs from the Sites reference table.
valid_site_ids = set(
    sites_clean["site_id"].dropna()
)


# Create a set of valid cement types from the CementTypes table.
valid_cement_types = set(
    cement_types_clean["cement_type"].dropna()
)


# Identify Operations records whose site IDs are not in the Sites table.
invalid_site_mask = (
    ~operations_clean["site_id"].isin(
        valid_site_ids
    )
)


# Identify Operations records whose cement types are not valid.
invalid_cement_mask = (
    ~operations_clean["cement_type"].isin(
        valid_cement_types
    )
)


# Display the number of invalid site records.
print(
    "Invalid site rows:",
    invalid_site_mask.sum()
)


# Display the number of invalid cement-type records.
print(
    "Invalid cement-type rows:",
    invalid_cement_mask.sum()
)


# Keep only records that contain both valid site IDs and cement types.
operations_clean = operations_clean.loc[
    ~invalid_site_mask
    & ~invalid_cement_mask
].copy()


Invalid site rows: 0
Invalid cement-type rows: 0


#Handle invalid negative values

In [84]:
# Define columns that should never contain negative values.
non_negative_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "silo_capacity"
]


# Check every non-negative field.
for column in non_negative_columns:

    # Count the negative values in the current column.
    negative_count = (
        operations_clean[column] < 0
    ).sum()

    # Display the result.
    print(
        f"{column}: {negative_count} negative values"
    )

    # Replace negative values with NaN because they are invalid.
    operations_clean.loc[
        operations_clean[column] < 0,
        column
    ] = np.nan


planned_pour_tonnes: 0 negative values
consumed_tonnes: 0 negative values
opening_inventory_tonnes: 0 negative values
deliveries_tonnes: 0 negative values
closing_inventory_tonnes: 0 negative values
rain_mm: 0 negative values
silo_capacity: 0 negative values


#Sort records before filling missing values

In [85]:
# Sort observations chronologically within each site and cement type.
# Correct sorting is important before interpolation and time-series modelling.
operations_clean = operations_clean.sort_values(
    by=[
        "site_id",
        "cement_type",
        "date"
    ]
)


 #Fill missing weather values

In [86]:
# Define weather columns that can reasonably be interpolated.
weather_columns = [
    "rain_mm",
    "avg_temp_c"
]


# Group the data by site and interpolate missing weather values over time.
operations_clean[weather_columns] = (
    operations_clean
    .groupby("site_id")[weather_columns]
    .transform(
        lambda group: group.interpolate(

            # Estimate missing values using a straight line
            # between surrounding observations.
            method="linear",

            # Fill missing values at both the beginning and end
            # of each site's sequence where possible.
            limit_direction="both"
        )
    )
)


#Fill missing operational values

In [87]:
# Define the main operational measurement columns.
operational_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes"
]


# Process every operational column separately.
for column in operational_columns:

    # Calculate the median for each site and cement-type combination.
    group_median = (
        operations_clean
        .groupby(
            [
                "site_id",
                "cement_type"
            ]
        )[column]
        .transform("median")
    )

    # Replace missing values with the appropriate group median.
    operations_clean[column] = (
        operations_clean[column]
        .fillna(group_median)
    )


#Use the Sites table as the source of truth for silo capacity

In [88]:
# Select only the site ID and silo capacity from the Sites table.
site_capacity_reference = (
    sites_clean[
        [
            "site_id",
            "silo_capacity"
        ]
    ]

    # Make sure each site appears only once.
    .drop_duplicates(
        subset=["site_id"]
    )

    # Rename the capacity field to avoid confusion during the merge.
    .rename(
        columns={
            "silo_capacity":
            "reference_silo_capacity"
        }
    )
)


# Join the trusted site capacity to every operational record.
operations_clean = operations_clean.merge(
    site_capacity_reference,
    on="site_id",
    how="left",

    # Confirm that many Operations rows map to one Sites row.
    validate="many_to_one"
)
#Check whether the two capacity values disagree:

# Compare the Operations capacity with the Sites reference capacity.
capacity_mismatch = (
    operations_clean["silo_capacity"]
    != operations_clean[
        "reference_silo_capacity"
    ]
)


# Display the number of differences.
print(
    "Silo-capacity mismatches:",
    capacity_mismatch.sum()
)
#Replace the Operations value:

# Use the Sites-table capacity when it is available.
# Otherwise, retain the original Operations value.
operations_clean["silo_capacity"] = (
    operations_clean[
        "reference_silo_capacity"
    ]
    .fillna(
        operations_clean["silo_capacity"]
    )
)


# Remove the temporary reference column.
operations_clean = operations_clean.drop(
    columns=[
        "reference_silo_capacity"
    ]
)


Silo-capacity mismatches: 0


#Validate the inventory balance

In [89]:
# Calculate what closing inventory should be according to the business rule:
# opening inventory + deliveries - consumed amount.
operations_clean[
    "calculated_closing_inventory"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    + operations_clean[
        "deliveries_tonnes"
    ]
    - operations_clean[
        "consumed_tonnes"
    ]
)


# Calculate the difference between reported and expected closing inventory.
operations_clean[
    "inventory_balance_difference"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ]
    - operations_clean[
        "calculated_closing_inventory"
    ]
).round(4)
#Define a rounding tolerance:

# Allow very small differences caused by decimal rounding.
inventory_tolerance = 0.011


# Identify records where the difference is larger than the tolerance.
inventory_mismatch_mask = (
    operations_clean[
        "inventory_balance_difference"
    ].abs()
    > inventory_tolerance
)


# Display the number of material mismatches.
print(
    "Material inventory mismatches:",
    inventory_mismatch_mask.sum()
)
#Correct only material errors:

# Replace the reported closing inventory only where
# the difference exceeds the accepted tolerance.
operations_clean.loc[
    inventory_mismatch_mask,
    "closing_inventory_tonnes"
] = operations_clean.loc[
    inventory_mismatch_mask,
    "calculated_closing_inventory"
]


Material inventory mismatches: 0


#Round tonnage measurements

In [90]:
# Define columns that should use two decimal places.
tonnage_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes"
]


# Round all tonnage measurements to two decimal places.
operations_clean[tonnage_columns] = (
    operations_clean[
        tonnage_columns
    ].round(2)
)


 #Flag inventory above silo capacity

In [91]:
# Create a flag for opening inventory exceeding silo capacity.
operations_clean[
    "opening_over_capacity_flag"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    > operations_clean[
        "silo_capacity"
    ]
).astype("int8")


# Create a flag for closing inventory exceeding silo capacity.
operations_clean[
    "closing_over_capacity_flag"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ]
    > operations_clean[
        "silo_capacity"
    ]
).astype("int8")
#Display the counts:

# Count opening-capacity exceptions.
print(
    "Opening inventory over capacity:",
    operations_clean[
        "opening_over_capacity_flag"
    ].sum()
)


# Count closing-capacity exceptions.
print(
    "Closing inventory over capacity:",
    operations_clean[
        "closing_over_capacity_flag"
    ].sum()
)


Opening inventory over capacity: 11427
Closing inventory over capacity: 11439


#Calculate available inventory

In [92]:
# Calculate the total cement available during the day.
operations_clean[
    "available_inventory_tonnes"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    + operations_clean[
        "deliveries_tonnes"
    ]
)


#Identify consumption above available stock

In [93]:
# Flag records where consumption is greater than
# the opening inventory plus deliveries.
operations_clean[
    "consumption_over_available_flag"
] = (
    operations_clean[
        "consumed_tonnes"
    ]
    > (
        operations_clean[
            "available_inventory_tonnes"
        ]
        + 0.01
    )
).astype("int8")


# Display the number of physically impossible consumption records.
print(
    "Consumption above available inventory:",
    operations_clean[
        "consumption_over_available_flag"
    ].sum()
)
#Correct only impossible values:

# Identify the rows that failed the availability check.
consumption_error_mask = (
    operations_clean[
        "consumption_over_available_flag"
    ] == 1
)


# Set consumption equal to available inventory
# when the original consumption exceeds physical availability.
operations_clean.loc[
    consumption_error_mask,
    "consumed_tonnes"
] = operations_clean.loc[
    consumption_error_mask,
    "available_inventory_tonnes"
]


Consumption above available inventory: 0


#Recalculate closing inventory

In [94]:
# Recalculate closing inventory after correcting any consumption errors.
operations_clean[
    "closing_inventory_tonnes"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    + operations_clean[
        "deliveries_tonnes"
    ]
    - operations_clean[
        "consumed_tonnes"
    ]
)

# Prevent small floating-point errors from creating negative stock.
operations_clean[
    "closing_inventory_tonnes"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ]
    .clip(lower=0)
    .round(2)
)


#Create business metrics

In [95]:
# Calculate the amount of planned cement demand that was not fulfilled.
operations_clean[
    "unmet_demand_tonnes"
] = (
    operations_clean[
        "planned_pour_tonnes"
    ]
    - operations_clean[
        "consumed_tonnes"
    ]
)

# Prevent negative unmet-demand values.
operations_clean[
    "unmet_demand_tonnes"
] = (
    operations_clean[
        "unmet_demand_tonnes"
    ]
    .clip(lower=0)
    .round(2)
)
#Create a stock-out flag:

# Mark records where closing inventory is zero.
operations_clean[
    "stockout_flag"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ] <= 0
).astype("int8")
#Calculate the pour-fulfilment rate:

# Calculate the percentage of planned pour demand that was fulfilled.
operations_clean[
    "pour_fulfilment_rate"
] = np.where(

    # Only calculate the ratio when planned demand is above zero.
    operations_clean[
        "planned_pour_tonnes"
    ] > 0,

    # Divide actual consumption by planned demand.
    operations_clean[
        "consumed_tonnes"
    ]
    / operations_clean[
        "planned_pour_tonnes"
    ],

    # Return NaN where planned demand is zero.
    np.nan
)


# Prevent negative values and round the rate.
operations_clean[
    "pour_fulfilment_rate"
] = (
    operations_clean[
        "pour_fulfilment_rate"
    ]
    .clip(lower=0)
    .round(4)
)


#Add site attributes

In [96]:
# Select descriptive site attributes.
site_attributes = (
    sites_clean[
        [
            "site_id",
            "region",
            "behavior"
        ]
    ]

    # Ensure each site appears only once.
    .drop_duplicates(
        subset=["site_id"]
    )
)


# Merge the site attributes into the Operations dataset.
operations_clean = operations_clean.merge(
    site_attributes,
    on="site_id",
    how="left",

    # Validate that every operational site maps to one reference record.
    validate="many_to_one"
)


#Remove temporary columns

In [97]:
# Define intermediate columns that are no longer required.
temporary_columns = [
    "calculated_closing_inventory",
    "inventory_balance_difference"
]


# Remove temporary columns.
# errors="ignore" prevents failure if a column does not exist.
operations_clean = operations_clean.drop(
    columns=temporary_columns,
    errors="ignore"
)


#Sort and reset the index

In [98]:
# Sort the final dataset into a consistent chronological order.
operations_clean = operations_clean.sort_values(
    by=[
        "site_id",
        "cement_type",
        "date"
    ]
)


# Replace the old row index with a clean sequential index.
operations_clean = operations_clean.reset_index(
    drop=True
)


#Final quality checks

In [99]:
# Display the dimensions of the cleansed dataset.
print(
    "Final shape:",
    operations_clean.shape
)


# Display the number of missing values in every column.
print(
    "\nMissing values:\n",
    operations_clean.isna().sum()
)


# Count completely duplicated rows.
print(
    "\nExact duplicates:",
    operations_clean.duplicated().sum()
)


# Count duplicated business keys.
print(
    "\nDuplicate business keys:",
    operations_clean.duplicated(
        subset=[
            "date",
            "site_id",
            "cement_type"
        ]
    ).sum()
)


# Display the minimum and maximum dates.
print(
    "\nDate range:",
    operations_clean["date"].min(),
    "to",
    operations_clean["date"].max()
)


# Count negative values in columns that must be non-negative.
print(
    "\nNegative non-weather values:",
    {
        column: int(
            (
                operations_clean[column] < 0
            ).sum()
        )
        for column in non_negative_columns
    }
)


Final shape: (32880, 20)

Missing values:
 date                                  0
site_id                               0
cement_type                           0
planned_pour_tonnes                   0
consumed_tonnes                       0
opening_inventory_tonnes              0
deliveries_tonnes                     0
closing_inventory_tonnes              0
rain_mm                               0
avg_temp_c                            0
silo_capacity                         0
opening_over_capacity_flag            0
closing_over_capacity_flag            0
available_inventory_tonnes            0
consumption_over_available_flag       0
unmet_demand_tonnes                   0
stockout_flag                         0
pour_fulfilment_rate               2993
region                                0
behavior                              0
dtype: int64



Exact duplicates: 0

Duplicate business keys: 0

Date range: 2022-01-01 00:00:00 to 2024-12-31 00:00:00

Negative non-weather values: {'planned_pour_tonnes': 0, 'consumed_tonnes': 0, 'opening_inventory_tonnes': 0, 'deliveries_tonnes': 0, 'closing_inventory_tonnes': 0, 'rain_mm': 0, 'silo_capacity': 0}


In [100]:
# Display the dimensions of the cleansed dataset.
print(
    "Final shape:",
    operations_clean.shape
)


# Display the number of missing values in every column.
print(
    "\nMissing values:\n",
    operations_clean.isna().sum()
)


# Count completely duplicated rows.
print(
    "\nExact duplicates:",
    operations_clean.duplicated().sum()
)


# Count duplicated business keys.
print(
    "\nDuplicate business keys:",
    operations_clean.duplicated(
        subset=[
            "date",
            "site_id",
            "cement_type"
        ]
    ).sum()
)


# Display the minimum and maximum dates.
print(
    "\nDate range:",
    operations_clean["date"].min(),
    "to",
    operations_clean["date"].max()
)


# Count negative values in columns that must be non-negative.
print(
    "\nNegative non-weather values:",
    {
        column: int(
            (
                operations_clean[column] < 0
            ).sum()
        )
        for column in non_negative_columns
    }
)


Final shape: (32880, 20)

Missing values:
 date                                  0
site_id                               0
cement_type                           0
planned_pour_tonnes                   0
consumed_tonnes                       0
opening_inventory_tonnes              0
deliveries_tonnes                     0
closing_inventory_tonnes              0
rain_mm                               0
avg_temp_c                            0
silo_capacity                         0
opening_over_capacity_flag            0
closing_over_capacity_flag            0
available_inventory_tonnes            0
consumption_over_available_flag       0
unmet_demand_tonnes                   0
stockout_flag                         0
pour_fulfilment_rate               2993
region                                0
behavior                              0
dtype: int64

Exact duplicates: 0

Duplicate business keys: 0

Date range: 2022-01-01 00:00:00 to 2024-12-31 00:00:00

Negative non-weather values: {'planned

#Add automated assertions

In [101]:
# Confirm that all dates are present.
assert operations_clean[
    "date"
].notna().all()


# Confirm that all site IDs are present.
assert operations_clean[
    "site_id"
].notna().all()


# Confirm that all cement types are present.
assert operations_clean[
    "cement_type"
].notna().all()


# Confirm that no duplicated business keys remain.
assert operations_clean.duplicated(
    subset=[
        "date",
        "site_id",
        "cement_type"
    ]
).sum() == 0


# Confirm that every site ID exists in the Sites reference table.
assert operations_clean[
    "site_id"
].isin(
    sites_clean["site_id"]
).all()


# Confirm that every cement type exists in the CementTypes table.
assert operations_clean[
    "cement_type"
].isin(
    cement_types_clean[
        "cement_type"
    ]
).all()


#Create the processed-data directory

In [102]:
# Import Path for safe, cross-platform file paths.
from pathlib import Path


# Build the path to the processed-data directory.
processed_directory = (
    Path.cwd().parent
    / "data"
    / "processed"
)


# Create the folder if it does not already exist.
processed_directory.mkdir(
    parents=True,
    exist_ok=True
)


 #Save the cleansed datasets as Parquet

#Save a CSV copy

In [103]:
# Save the Operations dataset as CSV for easy viewing and sharing.
operations_clean.to_csv(
    processed_directory
    / "cement_operations_clean.csv",
    index=False
)


 #Confirm the saved files

In [104]:
# Display the full folder path where the cleansed data was saved.
print(
    "Cleaned data saved to:",
    processed_directory.resolve()
)


# Display the files created in the processed-data folder.
for file_path in processed_directory.iterdir():
    print(file_path.name)


Cleaned data saved to: C:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\processed
.gitkeep
cement_operations_clean.csv
cement_operations_clean.parquet
cement_operations_merged.csv
cement_operations_merged.parquet
cement_types_clean.parquet
sites_clean.parquet


In [105]:
# Merge Operations with Sites
merged_data = operations_clean.merge(
    sites_clean[
        [
            "site_id",
            "region",
            "behavior",
            "silo_capacity"
        ]
    ],
    on="site_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_site")
)

# Validate the merge
print("Rows before merge:", len(operations_clean))
print("Rows after merge:", len(merged_data))
print("Missing site matches:", merged_data["region"].isna().sum())

Rows before merge: 32880
Rows after merge: 32880
Missing site matches: 0


In [106]:
print(merged_data.columns.tolist())
print(merged_data.head())

['date', 'site_id', 'cement_type', 'planned_pour_tonnes', 'consumed_tonnes', 'opening_inventory_tonnes', 'deliveries_tonnes', 'closing_inventory_tonnes', 'rain_mm', 'avg_temp_c', 'silo_capacity', 'opening_over_capacity_flag', 'closing_over_capacity_flag', 'available_inventory_tonnes', 'consumption_over_available_flag', 'unmet_demand_tonnes', 'stockout_flag', 'pour_fulfilment_rate', 'region', 'behavior', 'region_site', 'behavior_site', 'silo_capacity_site']
        date   site_id cement_type  planned_pour_tonnes  consumed_tonnes  \
0 2022-01-02  SITE_001       CEM_I                45.26            45.26   
1 2022-01-04  SITE_001       CEM_I                33.16            33.16   
2 2022-01-07  SITE_001       CEM_I                33.60            33.60   
3 2022-01-08  SITE_001       CEM_I                42.12            34.28   
4 2022-01-09  SITE_001       CEM_I                 0.00             0.00   

   opening_inventory_tonnes  deliveries_tonnes  closing_inventory_tonnes  \
0     

In [107]:
# Remove duplicate columns created by merging the Sites table
# after the site attributes had already been added.
duplicate_site_columns = [
    "region_site",
    "behavior_site",
    "silo_capacity_site"
]


# Drop only the duplicate columns.
# errors="ignore" prevents an error if any column is not present.
merged_data = merged_data.drop(
    columns=duplicate_site_columns,
    errors="ignore"
)


# Confirm the final dataset structure.
print("Final shape:", merged_data.shape)
print("Final columns:")
print(merged_data.columns.tolist())

Final shape: (32880, 20)
Final columns:
['date', 'site_id', 'cement_type', 'planned_pour_tonnes', 'consumed_tonnes', 'opening_inventory_tonnes', 'deliveries_tonnes', 'closing_inventory_tonnes', 'rain_mm', 'avg_temp_c', 'silo_capacity', 'opening_over_capacity_flag', 'closing_over_capacity_flag', 'available_inventory_tonnes', 'consumption_over_available_flag', 'unmet_demand_tonnes', 'stockout_flag', 'pour_fulfilment_rate', 'region', 'behavior']


In [108]:
# Confirm that there are no exact duplicate records.
print("Exact duplicates:", merged_data.duplicated().sum())

# Confirm that each date-site-cement combination appears only once.
print(
    "Duplicate business keys:",
    merged_data.duplicated(
        subset=["date", "site_id", "cement_type"]
    ).sum()
)

# Check for missing values in every column.
print("\nMissing values:")
print(merged_data.isna().sum())

# Confirm the final date range.
print(
    "\nDate range:",
    merged_data["date"].min(),
    "to",
    merged_data["date"].max()
)

Exact duplicates: 0
Duplicate business keys: 0

Missing values:
date                                  0
site_id                               0
cement_type                           0
planned_pour_tonnes                   0
consumed_tonnes                       0
opening_inventory_tonnes              0
deliveries_tonnes                     0
closing_inventory_tonnes              0
rain_mm                               0
avg_temp_c                            0
silo_capacity                         0
opening_over_capacity_flag            0
closing_over_capacity_flag            0
available_inventory_tonnes            0
consumption_over_available_flag       0
unmet_demand_tonnes                   0
stockout_flag                         0
pour_fulfilment_rate               2993
region                                0
behavior                              0
dtype: int64

Date range: 2022-01-01 00:00:00 to 2024-12-31 00:00:00


In [109]:
missing_rate = operations_clean[
    operations_clean["pour_fulfilment_rate"].isna()
]

print(
    missing_rate["planned_pour_tonnes"].value_counts()
)

planned_pour_tonnes
0.0    2993
Name: count, dtype: int64


# Quality Check:
# The remaining missing values occur only in pour_fulfilment_rate.
# These correspond to records where planned_pour_tonnes equals zero.
# Since a fulfilment rate cannot be calculated when no pour was planned,
# NaN is intentionally retained to represent a non-applicable value.

In [110]:
# ----------------------------------------------------------
# Save the final cleaned and integrated dataset
# ----------------------------------------------------------

# Save the dataset in Parquet format.
# Parquet is efficient, compressed, and preserves data types,
# making it suitable for analytics and machine learning.
merged_data.to_parquet(
    processed_directory / "cement_operations_merged.parquet",
    index=False
)

# Save a CSV version for easy viewing, sharing, and interoperability.
merged_data.to_csv(
    processed_directory / "cement_operations_merged.csv",
    index=False
)

# Display the output location.
print("Final merged datasets saved successfully.")
print("Location:", processed_directory.resolve())

# Display the files created in the processed-data directory.
print("\nGenerated files:")
for file_path in processed_directory.iterdir():
    print("-", file_path.name)

Final merged datasets saved successfully.
Location: C:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\processed

Generated files:
- .gitkeep
- cement_operations_clean.csv
- cement_operations_clean.parquet
- cement_operations_merged.csv
- cement_operations_merged.parquet
- cement_types_clean.parquet
- sites_clean.parquet


In [111]:
# Check for missing values in the raw Operations table.
print("Missing values in each column:")
print(operations.isnull().sum())

Missing values in each column:
date                            0
site_id                         0
cement_type                     0
planned_pour_tonnes             0
consumed_tonnes                 0
opening_inventory_tonnes        0
deliveries_tonnes               0
closing_inventory_tonnes        0
rain_mm                         0
avg_temp_c                      0
silo_capacity                   0
calculated_closing_inventory    0
inventory_difference            0
dtype: int64


In [112]:
print(operations_clean.shape)
print(operations_clean.columns.tolist())

(32880, 20)
['date', 'site_id', 'cement_type', 'planned_pour_tonnes', 'consumed_tonnes', 'opening_inventory_tonnes', 'deliveries_tonnes', 'closing_inventory_tonnes', 'rain_mm', 'avg_temp_c', 'silo_capacity', 'opening_over_capacity_flag', 'closing_over_capacity_flag', 'available_inventory_tonnes', 'consumption_over_available_flag', 'unmet_demand_tonnes', 'stockout_flag', 'pour_fulfilment_rate', 'region', 'behavior']


In [113]:
print(operations_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 20 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   date                             32880 non-null  datetime64[us]
 1   site_id                          32880 non-null  string        
 2   cement_type                      32880 non-null  string        
 3   planned_pour_tonnes              32880 non-null  float64       
 4   consumed_tonnes                  32880 non-null  float64       
 5   opening_inventory_tonnes         32880 non-null  float64       
 6   deliveries_tonnes                32880 non-null  float64       
 7   closing_inventory_tonnes         32880 non-null  float64       
 8   rain_mm                          32880 non-null  float64       
 9   avg_temp_c                       32880 non-null  float64       
 10  silo_capacity                    32880 non-null  int64         
 11  

In [114]:
# Save the final cleaned dataset as CSV.
operations_clean.to_csv(
    processed_directory / "cement_operations_clean.csv",
    index=False
)

# Save the final cleaned dataset as Parquet.
operations_clean.to_parquet(
    processed_directory / "cement_operations_clean.parquet",
    engine="pyarrow",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [115]:
# Save the final cleaned dataset as CSV.
operations_clean.to_csv(
    processed_directory / "cement_operations_clean.csv",
    index=False
)

# Save the final cleaned dataset as Parquet.
operations_clean.to_parquet(
    processed_directory / "cement_operations_clean.parquet",
    engine="pyarrow",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [116]:
print((processed_directory / "cement_operations_clean.csv").exists())
print((processed_directory / "cement_operations_clean.parquet").exists())

True
True
